# Create sensitivity-tagged Delta tables

Creates four HIPAA-themed Delta tables in the lakehouse with deterministic fake data and sets a `data-sensitivity` TBLPROPERTY on each.

Idempotent and non-destructive: each table is created only if it does not already exist (writes use `mode('errorifexists')` gated by a `tableExists` check). The `ALTER TABLE ... SET TBLPROPERTIES` step is re-applied each run and is safe (sets metadata only, no data change).

In [ ]:
import random
from datetime import date, datetime, timedelta
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, DateType, TimestampType
)

random.seed(42)

TABLES = [
    ('patients', 'Highly Confidential'),
    ('claims', 'Confidential'),
    ('appointments', 'General'),
    ('providers', 'Public'),
]

N_ROWS = 50

In [ ]:
FIRST_NAMES = [
    'Alice', 'Brian', 'Carol', 'David', 'Eva', 'Frank', 'Grace', 'Henry',
    'Iris', 'Jack', 'Karen', 'Leo', 'Mia', 'Nathan', 'Olivia', 'Peter',
    'Quinn', 'Rachel', 'Sam', 'Tina', 'Uma', 'Victor', 'Wendy', 'Xavier',
    'Yara', 'Zach',
]
LAST_NAMES = [
    'Adams', 'Brown', 'Clark', 'Davis', 'Evans', 'Foster', 'Garcia', 'Harris',
    'Iverson', 'Jones', 'King', 'Lewis', 'Martin', 'Nelson', 'Owens', 'Patel',
    'Quinn', 'Roberts', 'Smith', 'Taylor', 'Underwood', 'Vance', 'White',
    'Xu', 'Young', 'Zimmer',
]
DIAGNOSES = [
    'Hypertension', 'Type 2 Diabetes', 'Asthma', 'Migraine', 'Anemia',
    'Influenza', 'Bronchitis', 'Arthritis', 'Hyperlipidemia', 'GERD',
]
SPECIALTIES = [
    'Cardiology', 'Endocrinology', 'Pulmonology', 'Neurology', 'Hematology',
    'Family Medicine', 'Internal Medicine', 'Rheumatology', 'Gastroenterology',
]
CLINICS = [
    'Contoso General', 'Lakeside Health', 'Riverbend Clinic', 'Hillcrest Care',
    'Northgate Medical', 'Southport Wellness',
]
LOCATIONS = [
    'Bldg A Rm 101', 'Bldg A Rm 210', 'Bldg B Rm 305', 'Bldg C Rm 110',
    'Telehealth', 'Bldg B Rm 402',
]
VISIT_TYPES = ['New', 'Follow-up', 'Annual', 'Urgent', 'Telehealth']
CLAIM_STATUSES = ['Submitted', 'Paid', 'Denied', 'Pending']

In [ ]:
def make_patients(n):
    rows = []
    base_dob = date(1950, 1, 1)
    for i in range(n):
        pid = 'P{:05d}'.format(1000 + i)
        fn = random.choice(FIRST_NAMES)
        ln = random.choice(LAST_NAMES)
        dob = base_dob + timedelta(days=random.randint(0, 365 * 70))
        ssn = '{:03d}-{:02d}-{:04d}'.format(
            random.randint(100, 899), random.randint(10, 99), random.randint(1000, 9999)
        )
        mrn = 'MRN{:07d}'.format(random.randint(1000000, 9999999))
        dx = random.choice(DIAGNOSES)
        rows.append((pid, fn, ln, dob, ssn, mrn, dx))
    return rows

patients_schema = StructType([
    StructField('patient_id', StringType(), False),
    StructField('first_name', StringType(), True),
    StructField('last_name', StringType(), True),
    StructField('dob', DateType(), True),
    StructField('ssn', StringType(), True),
    StructField('mrn', StringType(), True),
    StructField('primary_diagnosis', StringType(), True),
])

In [ ]:
def make_providers(n):
    rows = []
    for i in range(n):
        pid = 'PR{:04d}'.format(500 + i)
        fn = random.choice(FIRST_NAMES)
        ln = random.choice(LAST_NAMES)
        full = 'Dr. {} {}'.format(fn, ln)
        sp = random.choice(SPECIALTIES)
        phone = '({:03d}) {:03d}-{:04d}'.format(
            random.randint(200, 999), random.randint(200, 999), random.randint(0, 9999)
        )
        clinic = random.choice(CLINICS)
        rows.append((pid, full, sp, phone, clinic))
    return rows

providers_schema = StructType([
    StructField('provider_id', StringType(), False),
    StructField('full_name', StringType(), True),
    StructField('specialty', StringType(), True),
    StructField('office_phone', StringType(), True),
    StructField('clinic_name', StringType(), True),
])

In [ ]:
def make_claims(n, patient_ids, provider_ids):
    rows = []
    base_date = date(2024, 1, 1)
    for i in range(n):
        cid = 'C{:07d}'.format(2000000 + i)
        pat = random.choice(patient_ids)
        prov = random.choice(provider_ids)
        cdate = base_date + timedelta(days=random.randint(0, 365))
        amt = round(random.uniform(50.0, 5000.0), 2)
        st = random.choice(CLAIM_STATUSES)
        rows.append((cid, pat, prov, cdate, amt, st))
    return rows

claims_schema = StructType([
    StructField('claim_id', StringType(), False),
    StructField('patient_id', StringType(), True),
    StructField('provider_id', StringType(), True),
    StructField('claim_date', DateType(), True),
    StructField('amount_usd', DoubleType(), True),
    StructField('status', StringType(), True),
])

In [ ]:
def make_appointments(n, patient_ids, provider_ids):
    rows = []
    base_dt = datetime(2024, 1, 1, 8, 0, 0)
    for i in range(n):
        aid = 'A{:07d}'.format(3000000 + i)
        prov = random.choice(provider_ids)
        pat = random.choice(patient_ids)
        offset_minutes = random.randint(0, 60 * 24 * 365)
        adt = base_dt + timedelta(minutes=offset_minutes)
        loc = random.choice(LOCATIONS)
        vt = random.choice(VISIT_TYPES)
        rows.append((aid, prov, pat, adt, loc, vt))
    return rows

appointments_schema = StructType([
    StructField('appointment_id', StringType(), False),
    StructField('provider_id', StringType(), True),
    StructField('patient_id', StringType(), True),
    StructField('appointment_date', TimestampType(), True),
    StructField('location', StringType(), True),
    StructField('visit_type', StringType(), True),
])

In [ ]:
patient_rows = make_patients(N_ROWS)
provider_rows = make_providers(N_ROWS)
patient_ids = [r[0] for r in patient_rows]
provider_ids = [r[0] for r in provider_rows]
claim_rows = make_claims(N_ROWS, patient_ids, provider_ids)
appointment_rows = make_appointments(N_ROWS, patient_ids, provider_ids)

datasets = {
    'patients': (patient_rows, patients_schema),
    'providers': (provider_rows, providers_schema),
    'claims': (claim_rows, claims_schema),
    'appointments': (appointment_rows, appointments_schema),
}

for name, (rows, schema) in datasets.items():
    if spark.catalog.tableExists(name):
        print('Skipping {}: table already exists (preserving existing data).'.format(name))
        continue
    df = spark.createDataFrame(rows, schema)
    (df.write
       .format('delta')
       .mode('errorifexists')
       .saveAsTable(name))
    print('Wrote table {} with {} rows'.format(name, df.count()))

In [ ]:
for table_name, sensitivity in TABLES:
    sql = (
        "ALTER TABLE " + table_name +
        " SET TBLPROPERTIES ('data-sensitivity' = '" + sensitivity + "')"
    )
    print(sql)
    spark.sql(sql)

In [ ]:
for table_name, _sensitivity in TABLES:
    print('=' * 60)
    print('DESCRIBE EXTENDED ' + table_name)
    print('=' * 60)
    spark.sql('DESCRIBE EXTENDED ' + table_name).show(n=200, truncate=False)

print('=' * 60)
print('SHOW TABLES')
print('=' * 60)
spark.sql('SHOW TABLES').show(truncate=False)